# SAMHSA cleaning pipeline

This notebook loads `data/samhsa_filtered_data.csv`, applies the cleaning steps we defined (drop empty columns, standardized location, address flags, trimmed text, stable IDs, phone parsing, missingness report, normalized multi-value fields, validation), and writes:

- `data/samhsa_clean.csv` — cleaned tabular data  
- `data/facilities.json` — array of records for the web app  

**Environment:** Use the venv in `python/`: `cd python && python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`. In VS Code / Jupyter, pick the interpreter **`python/.venv/bin/python`** or kernel **Python (BHA python/.venv)**.

**Paths:** `PROJECT_ROOT` is detected automatically (folder that contains `data/samhsa_filtered_data.csv`), so you can run this notebook from the repo root or from `python/notebooks/`.

## 1. Setup: imports and project paths

Import libraries and set paths to the raw CSV and output files. Uses `pandas` for tabular work.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

def find_project_root() -> Path:
    """Repo root = directory containing data/samhsa_filtered_data.csv."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "samhsa_filtered_data.csv").exists():
            return p
    raise FileNotFoundError(
        "Could not find data/samhsa_filtered_data.csv — run from inside the BHA-website-2.0 repo."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_CSV = DATA_DIR / "samhsa_filtered_data.csv"
CLEAN_CSV = DATA_DIR / "samhsa_clean.csv"
FACILITIES_JSON = DATA_DIR / "facilities.json"

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("RAW_CSV exists:", RAW_CSV.exists())

PROJECT_ROOT: /Users/sarthakchandervanshi/Desktop/Required/D Drive/Masters Data Science/AI Campus/BHA-website-2.0
RAW_CSV exists: True


## 2. Load the raw CSV

Read the file with UTF-8. This preserves SAMHSA comma-separated text fields inside quoted cells.

In [2]:
df = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]
print("rows:", len(df), "cols:", len(df.columns))
df.head(2)

rows: 1039 cols: 30


,city,state,zip,phone,intake1,intake2,intake1a,intake2a,facility_name,address,...,special_programs_groups,assessment_pretreatment,testing,recovery_support,education_counseling,smoking_policy,age_groups_accepted,language_services,vaping_policy,ancillary_services
0,Bear,DE,19701,302-836-2864,302-224-6800,,,,Westside Family Healthcare,404 Fox Hunt Drive,...,,Screening for tobacco use,"Laboratory testing, Metabolic syndrome monitoring",,Smoking / vaping / tobacco cessation counseling,Smoking not permitted,"Adults, Children / adolescents, Seniors, Young...",Sign language services for the deaf and hard o...,Vaping not permitted,"Case management service, Chronic disease / ill..."
1,Dover,DE,19904,302-678-3020,,,,,Delaware Guidance Services Children and Youth ...,103 Mont Blanc Boulevard,...,Children/adolescents with serious emotional di...,Screening for tobacco use,Metabolic syndrome monitoring,,,Smoking permitted without restriction,"Adults, Children / adolescents, Young adults",Sign language services for the deaf and hard o...,Vaping permitted without restriction,"Case management service, Family psychoeducatio..."


## 3. Drop columns that are completely empty

Remove `intake1a` and `intake2a` (100% empty in this extract) so downstream exports stay lean.

In [3]:
EMPTY_DROP = ["intake1a", "intake2a"]
present = [c for c in EMPTY_DROP if c in df.columns]
df = df.drop(columns=present, errors="ignore")
print("Dropped:", present or "(none — columns missing)")

Dropped: ['intake1a', 'intake2a']


## 4. Standardize `state`, `zip`, and `city`

- **state**: two-letter uppercase  
- **zip**: digits only, left-padded to 5 characters (string)  
- **city**: strip; **city_display** uses title case for UI labels

In [4]:
def standardize_zip(z: str) -> str:
    if pd.isna(z) or z == "":
        return ""
    digits = re.sub(r"\D", "", str(z))[:5]
    if len(digits) < 5:
        return digits.zfill(5) if digits else ""
    return digits


def standardize_state(s: str) -> str:
    if pd.isna(s) or s == "":
        return ""
    return str(s).strip().upper()[:2]


df["city_raw"] = df["city"].astype(str).str.strip()
df["city"] = df["city_raw"]
df["city_display"] = df["city"].apply(lambda x: str(x).strip().title() if str(x).strip() else "")
df["state"] = df["state"].apply(standardize_state)
df["zip"] = df["zip"].apply(standardize_zip)

allowed_states = {"DE", "NJ", "NY", "PA"}
bad_state = ~df["state"].isin(allowed_states | {""})
print("rows with unexpected state:", int(bad_state.sum()))
if bad_state.any():
    print(df.loc[bad_state, ["facility_name", "state"]].head(10).to_string())

rows with unexpected state: 0


## 5. Address quality: flags and placeholder handling

Flag empty addresses and SAMHSA placeholder `- - -`. **`address_clean`** is empty when the street was withheld; use it for geocoding logic.

In [5]:
PLACEHOLDER = "- - -"


def clean_address(a: str) -> str:
    if pd.isna(a):
        return ""
    s = str(a).strip()
    if s == PLACEHOLDER:
        return ""
    return s


df["address"] = df["address"].astype(str).apply(
    lambda x: x.strip() if x.strip() not in ("nan", "None") else ""
)
df["address_clean"] = df["address"].apply(clean_address)
df["address_is_empty"] = df["address_clean"] == ""
df["address_was_placeholder"] = df["address"].astype(str).str.strip() == PLACEHOLDER

print("address_is_empty:", int(df["address_is_empty"].sum()))
print("address_was_placeholder:", int(df["address_was_placeholder"].sum()))

address_is_empty: 66
address_was_placeholder: 41


## 6. Trim whitespace on all string columns

Strip leading/trailing spaces on every text field so filters and exports behave consistently.

In [6]:
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        df[col] = df[col].replace({"nan": "", "None": ""})

## 7. Add a stable `facility_id`

Computed **after** location and address cleanup so the key reflects normalized `zip`/`state` and trimmed `address`. SHA-256 hex, first 16 characters.

In [7]:
def make_facility_id(row: pd.Series) -> str:
    parts = [
        str(row.get("facility_name", "")).strip(),
        str(row.get("address", "")).strip(),
        str(row.get("zip", "")).strip(),
        str(row.get("state", "")).strip(),
    ]
    key = "|".join(parts)
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]


df.insert(0, "facility_id", df.apply(make_facility_id, axis=1))
dup_ids = df["facility_id"].duplicated().sum()
print("duplicate facility_id count:", int(dup_ids))
assert dup_ids == 0, "Collision in facility_id — inspect rows"

duplicate facility_id count: 0


## 8. Phone: main number and extension

Parse `phone` into **`phone_main`** (formatted when 10 digits) and **`phone_extension`** when patterns like `x1234` appear.

In [8]:
EXT_RE = re.compile(r"(?i)\s*x\s*([0-9]+)\s*$")


def split_phone(raw: str) -> tuple[str, str, str]:
    """Returns (original_display, phone_main_formatted, extension)."""
    if pd.isna(raw) or str(raw).strip() == "":
        return "", "", ""
    s = str(raw).strip()
    ext = ""
    m = EXT_RE.search(s)
    if m:
        ext = m.group(1)
        s = EXT_RE.sub("", s).strip()
    digits = re.sub(r"\D", "", s)
    if len(digits) == 10:
        main_fmt = f"{digits[:3]}-{digits[3:6]}-{digits[6:]}"
    else:
        main_fmt = s
    return raw.strip(), main_fmt, ext


split = df["phone"].apply(split_phone)
df["phone_display"] = split.apply(lambda t: t[0])
df["phone_main"] = split.apply(lambda t: t[1])
df["phone_extension"] = split.apply(lambda t: t[2])
print("rows with extension:", int((df["phone_extension"] != "").sum()))
df[["phone", "phone_main", "phone_extension"]].head(6)

rows with extension: 156


,phone,phone_main,phone_extension
0,302-836-2864,302-836-2864,
1,302-678-3020,302-678-3020,
2,302-674-2380 x1218,302-674-2380,1218
3,302-730-0720,302-730-0720,
4,302-678-9962,302-678-9962,
5,302-747-1145,302-747-1145,


## 9. Document missingness (per-column report)

Treat empty string as missing. Use this to see sparse domains (do not assume blank means “no service”).

In [9]:
def is_blank(x) -> bool:
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in ("none", "nan")


missing_pct = {}
for col in df.columns:
    missing_pct[col] = df[col].apply(is_blank).mean() * 100

miss_df = (
    pd.Series(missing_pct)
    .sort_values(ascending=False)
    .rename("pct_missing")
    .round(2)
    .to_frame()
)
miss_df.head(20)

,pct_missing
license_certification,88.45
phone_extension,84.99
intake2,81.14
payment_assistance,59.87
testing,57.75
recovery_support,57.27
emergency_services,51.68
intake1,49.66
education_counseling,38.79
language_services,26.85


## 10. Normalize multi-value SAMHSA strings

Split on commas, trim, drop duplicate segments (case-insensitive), sort alphabetically, join with ` | `. Original columns stay unchanged; normalized values are in **`*_norm`** columns.

In [10]:
MULTI_COLS = [
    "type_of_care",
    "service_setting",
    "treatment_approaches",
    "payment_funding",
    "special_programs_groups",
    "emergency_services",
    "ancillary_services",
    "language_services",
    "age_groups_accepted",
    "pharmacotherapies",
]


def normalize_multivalue(text: str) -> str:
    if pd.isna(text) or str(text).strip() == "":
        return ""
    parts = [p.strip() for p in str(text).split(",")]
    parts = [p for p in parts if p]
    seen: set[str] = set()
    unique: list[str] = []
    for p in parts:
        k = p.lower()
        if k not in seen:
            seen.add(k)
            unique.append(p)
    unique.sort(key=str.lower)
    return " | ".join(unique)


for c in MULTI_COLS:
    if c in df.columns:
        df[c + "_norm"] = df[c].apply(normalize_multivalue)

print("Added *_norm columns:", [c for c in df.columns if c.endswith("_norm")])

Added *_norm columns: ['type_of_care_norm', 'service_setting_norm', 'treatment_approaches_norm', 'payment_funding_norm', 'special_programs_groups_norm', 'emergency_services_norm', 'ancillary_services_norm', 'language_services_norm', 'age_groups_accepted_norm', 'pharmacotherapies_norm']


## 11. Validation checks

Assert unique `facility_id` and that non-empty ZIPs are five digits. Lists rows if any ZIP fails.

In [11]:
assert df["facility_id"].is_unique, "facility_id must be unique"

zip_bad = df["zip"].apply(lambda z: z != "" and len(z) != 5)
print("ZIP not length 5 (non-empty):", int(zip_bad.sum()))
if zip_bad.any():
    print(df.loc[zip_bad, ["facility_name", "zip"]].to_string())

assert df.loc[df["zip"] != "", "zip"].str.match(r"^\d{5}$").all(), "ZIP validation failed"

print("OK: validation passed for current extract.")

ZIP not length 5 (non-empty): 0
OK: validation passed for current extract.


## 12. Slim export schema

One column per concept for **`samhsa_clean.csv` / `facilities.json`**: `city` and `address` are final strings; **`phone`** is the single SAMHSA line (may include `x` extension in text). **`phone_main` / `phone_extension`** are dropped here (redundant with `phone`). **`address_is_empty` / `address_was_placeholder`** are dropped — treat empty `address` as missing. **`_*_norm`** columns are dropped (keep original SAMHSA text for filters). Staging columns (`city_raw`, raw `address`, `phone_display`) are removed.

In [12]:
_norm_cols = [c for c in df.columns if c.endswith("_norm")]
df_export = df.copy()
df_export = df_export.drop(columns=["city_raw", "city"], errors="ignore")
df_export = df_export.rename(columns={"city_display": "city"})
df_export = df_export.drop(columns=["address"], errors="ignore")
df_export = df_export.rename(columns={"address_clean": "address"})
df_export = df_export.drop(columns=["phone_display"] + _norm_cols, errors="ignore")
# Phone: keep SAMHSA `phone` only (includes "x1234" in the string when present); drop parsed duplicates.
df_export = df_export.drop(
    columns=["phone_main", "phone_extension"],
    errors="ignore",
)
# Address: empty `address` implies missing; drop redundant boolean flags.
df_export = df_export.drop(
    columns=["address_is_empty", "address_was_placeholder"],
    errors="ignore",
)
print("df_export shape:", df_export.shape)
print("columns:", list(df_export.columns))

df_export shape: (1039, 29)
columns: ['facility_id', 'state', 'zip', 'phone', 'intake1', 'intake2', 'facility_name', 'type_of_care', 'service_setting', 'facility_type', 'pharmacotherapies', 'treatment_approaches', 'emergency_services', 'facility_operation', 'license_certification', 'payment_funding', 'payment_assistance', 'special_programs_groups', 'assessment_pretreatment', 'testing', 'recovery_support', 'education_counseling', 'smoking_policy', 'age_groups_accepted', 'language_services', 'vaping_policy', 'ancillary_services', 'city', 'address']


## 13. Build JSON records for the website

Each row becomes one object from **`df_export`** (slim schema). Empty values are stored as empty strings for simple JS consumption.

In [13]:
def row_to_json_safe(row: pd.Series) -> dict:
    out = {}
    for col in df_export.columns:
        v = row[col]
        if pd.isna(v):
            out[col] = ""
        elif isinstance(v, (bool, np.bool_)):
            out[col] = bool(v)
        elif isinstance(v, str):
            out[col] = v
        else:
            out[col] = str(v)
    return out


records = [row_to_json_safe(row) for _, row in df_export.iterrows()]
print("records:", len(records), "fields per record:", len(records[0]) if records else 0)

records: 1039 fields per record: 29


## 14. Write `samhsa_clean.csv` and `facilities.json`

Write **`df_export`** (slim schema) to `data/`. Re-run after updating the raw CSV.

In [14]:
df_export.to_csv(CLEAN_CSV, index=False)
print("Wrote:", CLEAN_CSV)

with open(FACILITIES_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print("Wrote:", FACILITIES_JSON)

Wrote: /Users/sarthakchandervanshi/Desktop/Required/D Drive/Masters Data Science/AI Campus/BHA-website-2.0/data/samhsa_clean.csv
Wrote: /Users/sarthakchandervanshi/Desktop/Required/D Drive/Masters Data Science/AI Campus/BHA-website-2.0/data/facilities.json


## 15. Verify `samhsa_clean.csv`

Re-read the CSV from disk and show the first few rows (sanity check that the export round-trips).

In [15]:
df_csv_check = pd.read_csv(CLEAN_CSV, dtype=str, keep_default_na=False)
print("samhsa_clean.csv rows:", len(df_csv_check), "cols:", len(df_csv_check.columns))
with pd.option_context(
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", 40,
):
    display(df_csv_check.head())

samhsa_clean.csv rows: 1039 cols: 29


,facility_id,state,zip,phone,intake1,intake2,facility_name,type_of_care,service_setting,facility_type,pharmacotherapies,treatment_approaches,emergency_services,facility_operation,license_certification,payment_funding,payment_assistance,special_programs_groups,assessment_pretreatment,testing,recovery_support,education_counseling,smoking_policy,age_groups_accepted,language_services,vaping_policy,ancillary_services,city,address
0,a666465616d75f72,DE,19701,302-836-2864,302-224-6800,,Westside Family Healthcare,"Mental health treatment, Substance u...",Outpatient,Outpatient mental health facility,"Nicotine replacement, Non-nicotine s...","Cognitive behavioral therapy, Dialec...",,Private non-profit organization,Federally Qualified Health Center,"Cash or self-payment, Federal milita...",Sliding fee scale,,Screening for tobacco use,"Laboratory testing, Metabolic syndro...",,Smoking / vaping / tobacco cessation...,Smoking not permitted,"Adults, Children / adolescents, Seni...",Sign language services for the deaf ...,Vaping not permitted,"Case management service, Chronic dis...",Bear,404 Fox Hunt Drive
1,177f8ffd03bf80fe,DE,19904,302-678-3020,,,Delaware Guidance Services Children ...,Mental health treatment,Outpatient,Outpatient mental health facility,,"Abnormal involuntary movement scale,...","Crisis intervention team, Psychiatri...",Private non-profit organization,,"Cash or self-payment, Community Ment...","Payment assistance available, Slidin...",Children/adolescents with serious em...,Screening for tobacco use,Metabolic syndrome monitoring,,,Smoking permitted without restriction,"Adults, Children / adolescents, Youn...",Sign language services for the deaf ...,Vaping permitted without restriction,"Case management service, Family psyc...",Dover,103 Mont Blanc Boulevard
2,dcca9bbdffe68d10,DE,19901,302-674-2380 x1218,302-674-2380 x1239,,Mind and Body Consortium,"Mental health treatment, Substance u...",Outpatient,Outpatient mental health facility,Antipsychotics used in treatment of ...,"Cognitive behavioral therapy, Couple...",,Private for-profit organization,Mental health clinic or mental healt...,"Cash or self-payment, Federal milita...",,"Active duty military, Children/adole...",,Laboratory testing,,,Smoking permitted in designated area,"Adults, Children / adolescents, Seni...","Other languages excluding Spanish, S...",Vaping permitted in designated area,"Court-ordered outpatient treatment, ...",Dover,156 South State Street
3,c5095beae3bfca67,DE,19904,302-730-0720,,,New Behavioral Network,"Mental health treatment, Treatment f...",Outpatient,Multi-setting mental health facility,,"Cognitive behavioral therapy, Couple...",Crisis intervention team,Private for-profit organization,Mental health clinic or mental healt...,"Cash or self-payment, Medicaid, Priv...",,"Active duty military, Children/adole...",,,,,Smoking permitted without restriction,"Adults, Children / adolescents, Seni...",,Vaping permitted without restriction,"Case management service, Court-order...",Dover,1575 Mckee Road Suite 201
4,de6fe34c722da94b,DE,19904,302-678-9962,,,Psychotherapeutic Services Inc,"Mental health treatment, Substance u...",Residential / 24-hour residential,Residential treatment center for adults,Antipsychotics used in treatment of ...,"Activity therapy, Cognitive behavior...",,Private for-profit organization,,"Cash or self-payment, Medicaid, Medi...",,Clients with co-occurring mental and...,Screening for tobacco use,Laboratory testing,"Housing services, Mentoring / peer s...",Smoking / vaping / tobacco cessation...,Smoking permitted in designated area,"Adults, Seniors, Young adults",,Vaping permitted in designated area,"Case management service, Court-order...",Dover,942 Walker Road Suite B


## 16. Verify `facilities.json`

Load the JSON array and preview the first records as a table.

In [16]:
with open(FACILITIES_JSON, encoding="utf-8") as f:
    fac_list = json.load(f)
print("facilities.json records:", len(fac_list))
pd.DataFrame(fac_list).head()

facilities.json records: 1039


,facility_id,state,zip,phone,intake1,intake2,facility_name,type_of_care,service_setting,facility_type,...,testing,recovery_support,education_counseling,smoking_policy,age_groups_accepted,language_services,vaping_policy,ancillary_services,city,address
0,a666465616d75f72,DE,19701,302-836-2864,302-224-6800,,Westside Family Healthcare,"Mental health treatment, Substance use treatme...",Outpatient,Outpatient mental health facility,...,"Laboratory testing, Metabolic syndrome monitoring",,Smoking / vaping / tobacco cessation counseling,Smoking not permitted,"Adults, Children / adolescents, Seniors, Young...",Sign language services for the deaf and hard o...,Vaping not permitted,"Case management service, Chronic disease / ill...",Bear,404 Fox Hunt Drive
1,177f8ffd03bf80fe,DE,19904,302-678-3020,,,Delaware Guidance Services Children and Youth ...,Mental health treatment,Outpatient,Outpatient mental health facility,...,Metabolic syndrome monitoring,,,Smoking permitted without restriction,"Adults, Children / adolescents, Young adults",Sign language services for the deaf and hard o...,Vaping permitted without restriction,"Case management service, Family psychoeducatio...",Dover,103 Mont Blanc Boulevard
2,dcca9bbdffe68d10,DE,19901,302-674-2380 x1218,302-674-2380 x1239,,Mind and Body Consortium,"Mental health treatment, Substance use treatme...",Outpatient,Outpatient mental health facility,...,Laboratory testing,,,Smoking permitted in designated area,"Adults, Children / adolescents, Seniors, Young...","Other languages excluding Spanish, Spanish",Vaping permitted in designated area,"Court-ordered outpatient treatment, Diet and e...",Dover,156 South State Street
3,c5095beae3bfca67,DE,19904,302-730-0720,,,New Behavioral Network,"Mental health treatment, Treatment for co-occu...",Outpatient,Multi-setting mental health facility,...,,,,Smoking permitted without restriction,"Adults, Children / adolescents, Seniors, Young...",,Vaping permitted without restriction,"Case management service, Court-ordered outpati...",Dover,1575 Mckee Road Suite 201
4,de6fe34c722da94b,DE,19904,302-678-9962,,,Psychotherapeutic Services Inc,"Mental health treatment, Substance use treatme...",Residential / 24-hour residential,Residential treatment center for adults,...,Laboratory testing,"Housing services, Mentoring / peer support",Smoking / vaping / tobacco cessation counseling,Smoking permitted in designated area,"Adults, Seniors, Young adults",,Vaping permitted in designated area,"Case management service, Court-ordered outpati...",Dover,942 Walker Road Suite B
